# 02: Loading & Filtering the CellChat LR Pair Database

scCChain uses the [CellChatDB](https://github.com/jinworks/CellChat/tree/main/data)
(Jin et al., 2021) as the default ligand-receptor (LR) pair database. The database
is bundled as CSV files for human and mouse and exposed through `LRPairDB` — a
lightweight container of `LRPairRecord`s that stores LR pair names, ligand and
receptor gene symbols (which may be multi-subunit), and pathway annotations.

The database can be filtered to include only user-selected LR pairs, e.g., pairs
corresponding to secreted signaling. In principle, alternative and custom databases
can also be used, provided the required information (ligand symbols, receptor symbols,
and optionally pathway names) can be extracted.

We cover:
1. Loading the full CellChat database
2. Filtering by communication type
3. Filtering by pathway
4. Filtering by specific LR pairs
5. Loading the mouse database
6. Loading a custom LR pair database from CSV
7. Merging custom and CellChat databases

In [ ]:
using Pkg; Pkg.activate("../"); Pkg.instantiate()
using ScCChain, DataFrames, Printf

## Load the full database

Bundled CellChatDB CSVs are available for human and mouse. The default is human.
Each `LRPairRecord` contains the LR pair name (formatted as
`"ligand(s)—receptor(s)"`), ligand and receptor gene symbols, and the
pathway annotation.

In [ ]:
db = load_lrpair_db()
@printf("LR pairs: %d | Ligand genes: %d | Receptor genes: %d\n",
    n_lrpairs(db), length(all_ligands(db)), length(all_receptors(db)))

In [ ]:
first(to_dataframe(db), 5)

## Filter by communication type

The CellChat annotation column categorizes LR pairs into four communication types:
`"Secreted Signaling"`, `"Cell-Cell Contact"`, `"ECM-Receptor"`, and `"Non-protein Signaling"`.
Filtering to a specific type is useful, e.g., to restrict analysis to secreted signaling only.

In [ ]:
db_secreted = load_lrpair_db(; communication_type="Secreted Signaling")
@printf("Secreted Signaling LR pairs: %d\n", n_lrpairs(db_secreted))

## Filter by pathway

All filters are applied sequentially with AND logic. Here we select
secreted-signaling LR pairs from specific pathways.

In [ ]:
db_tgfb = load_lrpair_db(;
    communication_type="Secreted Signaling",
    pathways="TGFb",
)
@printf("TGFb (secreted) LR pairs: %d\n", n_lrpairs(db_tgfb))
first(to_dataframe(db_tgfb), 5)

In [ ]:
db_multi_pw = load_lrpair_db(;
    communication_type="Secreted Signaling",
    pathways=["TGFb", "WNT", "CXCL"],
)
@printf("TGFb + WNT + CXCL (secreted) LR pairs: %d\n", n_lrpairs(db_multi_pw))
first(to_dataframe(db_multi_pw), 10)

## Filter by specific LR pairs

In [ ]:
db_specific = load_lrpair_db(;
    communication_type="Secreted Signaling",
    lrpairs=["CXCL12—CXCR4", "TGFB1—TGFBR2_TGFBR1"],
)
to_dataframe(db_specific)

## Mouse database

In [ ]:
db_mouse = load_lrpair_db(; species="mouse")
@printf("Mouse LR pairs: %d\n", n_lrpairs(db_mouse))

## Custom LR pair database

In addition to CellChatDB, scCChain supports loading custom LR pair databases from
CSV (or Excel) files via `load_custom_lrpair_db`. The CSV must contain at least a
`ligand` and a `receptor` column. Multi-subunit complexes are represented as
comma-separated gene symbols within a single cell. Optional columns for LR pair
names (`interaction_name`) and pathway names can be mapped via keyword arguments.
If no name column is provided, names are auto-generated as `"ligand(s)—receptor(s)"`.

We use a small example CSV bundled at `data/examples/custom_LRpair_db/custom_lrpairs.csv`.

In [ ]:
custom_csv = joinpath(@__DIR__, "..", "data", "examples", "custom_LRpair_db", "custom_lrpairs.csv")
println("Custom CSV path: ", custom_csv)
println()
println(read(custom_csv, String))

In [ ]:
db_custom = load_custom_lrpair_db(
    custom_csv;
    lrpair_name_col=:interaction_name,
    pathway_col=:pathway,
)
@printf("Custom LR pairs: %d\n", n_lrpairs(db_custom))
to_dataframe(db_custom)

If your CSV uses non-default column names, map them via keyword arguments:

In [ ]:
# Example with explicit column mapping (same file, but showing the syntax):
db_custom2 = load_custom_lrpair_db(
    custom_csv;
    ligand_col=:ligand,
    receptor_col=:receptor,
    lrpair_name_col=:interaction_name,
    pathway_col=:pathway,
    species="human",
)
@printf("Custom DB (explicit mapping): %d LR pairs\n", n_lrpairs(db_custom2))

## Merging custom and CellChat databases

Custom databases can be merged with CellChatDB (or any other `LRPairDB`) using
`merge_lrpair_dbs`. Duplicates (by LR pair name) are resolved by keeping the entry
from the last database in argument order.

Alternatively, pass the `extend` keyword to `load_custom_lrpair_db` to merge
in a single call.

In [ ]:
# Method 1: explicit merge
db_cellchat = load_lrpair_db(; communication_type="Secreted Signaling")
db_merged = merge_lrpair_dbs(db_cellchat, db_custom)
@printf("CellChat (secreted): %d | Custom: %d | Merged: %d\n",
    n_lrpairs(db_cellchat), n_lrpairs(db_custom), n_lrpairs(db_merged))

In [ ]:
# Method 2: extend keyword (loads custom CSV and merges with existing DB in one step)
db_extended = load_custom_lrpair_db(
    custom_csv;
    lrpair_name_col=:interaction_name,
    pathway_col=:pathway,
    extend=db_cellchat,
)
@printf("Extended DB: %d LR pairs\n", n_lrpairs(db_extended))